## Validate Pipeline Output

### Load Pipeline Tables

In [0]:
CATALOG = "workspace"
SCHEMA = "hrm6131_landing"

BRONZE_PROFILES = f"{CATALOG}.{SCHEMA}.bronze_profiles"
BRONZE_DAY1 = f"{CATALOG}.{SCHEMA}.bronze_day1"
BRONZE_DAY2 = f"{CATALOG}.{SCHEMA}.bronze_day2"

SILVER_TICKETS = f"{CATALOG}.{SCHEMA}.silver_tickets"

GOLD_TEAM_PERFORMANCE = f"{CATALOG}.{SCHEMA}.gold_team_performance"
GOLD_AGENT_PERFORMANCE = f"{CATALOG}.{SCHEMA}.gold_agent_performance"
GOLD_QUALITY_COMPLIANCE = f"{CATALOG}.{SCHEMA}.gold_quality_compliance"
GOLD_DAY2_CARRYOVER = f"{CATALOG}.{SCHEMA}.gold_day2_carryover"

print("Configuration loaded successfully.")

Configuration loaded successfully.


In [0]:
bronze_profiles = spark.table(BRONZE_PROFILES)
bronze_day1 = spark.table(BRONZE_DAY1)
bronze_day2 = spark.table(BRONZE_DAY2)

silver = spark.table(SILVER_TICKETS)

team = spark.table(GOLD_TEAM_PERFORMANCE)
agent = spark.table(GOLD_AGENT_PERFORMANCE)
quality = spark.table(GOLD_QUALITY_COMPLIANCE)
carryover = spark.table(GOLD_DAY2_CARRYOVER)

print("All tables loaded successfully.")

All tables loaded successfully.


## Validate Row Counts

In [0]:
print("Bronze Profiles :", bronze_profiles.count())
print("Bronze Day 1    :", bronze_day1.count())
print("Bronze Day 2    :", bronze_day2.count())

print("Silver Tickets  :", silver.count())

print("Team Report     :", team.count())
print("Agent Report    :", agent.count())
print("Quality Report  :", quality.count())
print("Carry-over      :", carryover.count())

Bronze Profiles : 42
Bronze Day 1    : 123
Bronze Day 2    : 86
Silver Tickets  : 77
Team Report     : 16
Agent Report    : 77
Quality Report  : 40
Carry-over      : 4


## Check Duplicate Tickets

In [0]:
duplicates = (
    silver
    .groupBy("ticket_id")
    .count()
    .filter("count > 1")
)

print("Duplicate Tickets :", duplicates.count())

display(duplicates)

Duplicate Tickets : 0


ticket_id,count


## Check Null Values

In [0]:
from pyspark.sql import functions as F

silver.select(
    [
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in [
            "ticket_id",
            "agent_id",
            "status",
            "resolution_time_minutes"
        ]
    ]
).display()

ticket_id,agent_id,status,resolution_time_minutes
0,0,0,0


## Resolution Quality Rule Validation


In [0]:
from pyspark.sql import functions as F

print("Checking resolution quality rule")

invalid_resolution = silver.filter(
    (F.col("status") == "Resolved") &
    (F.col("resolution_time_minutes") <= 15)
)

print(
    "Resolved tickets violating >15 min rule:",
    invalid_resolution.count()
)

display(invalid_resolution)

Checking resolution quality rule
Resolved tickets violating >15 min rule: 0


agent_id,ticket_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file,agent_name,role,team_lead_id,hours,minutes,seconds,resolution_time_minutes


## Team Lead Scope Validation

In [0]:
print("Checking Team Lead scope")

invalid_team_leads = silver.filter(
    ~F.col("team_lead_id").isin(
        "TL01",
        "TL02",
        "TL03",
        "TL04",
        "TL05",
        "TL06",
        "TL07",
        "TL08"
    )
)

print(
    "Out-of-scope Team Leads:",
    invalid_team_leads.count()
)

display(invalid_team_leads)

Checking Team Lead scope
Out-of-scope Team Leads: 0


agent_id,ticket_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file,agent_name,role,team_lead_id,hours,minutes,seconds,resolution_time_minutes


## Day-2 Carry-over Validation

In [0]:
print("Checking Day-2 carry-over rule")

print("Checking Day-2 carry-over rule")

day1_success_agents = (
    silver
    .filter(
        (F.col("day") == 1) &
        (F.col("status") == "Resolved") &
        (F.col("resolution_time_minutes") > 15)
    )
    .select("agent_id")
    .distinct()
)


invalid_carryover = (
    carryover
    .join(
        day1_success_agents,
        "agent_id",
        "inner"
    )
)

print(
    "Day-2 agents incorrectly carried over:",
    invalid_carryover.count()
)

display(invalid_carryover)

Checking Day-2 carry-over rule
Checking Day-2 carry-over rule
Day-2 agents incorrectly carried over: 0


agent_id,ticket_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file,agent_name,role,team_lead_id,hours,minutes,seconds,resolution_time_minutes


## Validate Silver Layer

In [0]:
display(
    silver.select(
        "ticket_id",
        "agent_id",
        "status",
        "resolution_time",
        "resolution_time_minutes",
        "day"
    )
)

ticket_id,agent_id,status,resolution_time,resolution_time_minutes,day
TKT00001,A001,Resolved,0h 35m 45s,36,1
TKT00002,A001,Resolved,0h 22m 10s,22,1
TKT00061,A022,Resolved,0h 44m 0s,44,1
TKT00069,A025,Resolved,0h 26m 0s,26,1
TKT00094,A033,Resolved,0h 19m 45s,20,1
TKT00011,A004,Resolved,0h 33m 15s,33,1
TKT00045,A016,Resolved,0h 36m 0s,36,1
TKT00072,A026,Resolved,0h 23m 0s,23,1
TKT00086,A031,Resolved,0h 29m 0s,29,1
TKT00096,A034,Resolved,0h 25m 15s,25,1


## Validate Gold Layer

In [0]:
print("Team Performance Report")

display(team)

Team Performance Report


day,team_lead_id,tickets_resolved,avg_resolution_time
1,TL01,10,30.9
1,TL02,10,31.4
1,TL03,10,32.0
1,TL04,9,26.56
1,TL05,10,31.6
1,TL06,10,29.4
1,TL07,10,27.8
1,TL08,4,25.25
2,TL01,7,25.14
2,TL02,8,26.25


In [0]:
print("Agent Performance Report")

display(agent)

Agent Performance Report


day,agent_id,agent_name,team_lead_id,tickets_resolved,avg_resolution_time
1,A001,Agent_A001,TL01,3,26.0
1,A002,Agent_A002,TL01,2,43.5
1,A003,Agent_A003,TL01,2,22.5
1,A004,Agent_A004,TL01,2,27.0
1,A005,Agent_A005,TL01,1,45.0
1,A006,Agent_A006,TL02,2,23.0
1,A007,Agent_A007,TL02,2,44.0
1,A008,Agent_A008,TL02,2,25.0
1,A009,Agent_A009,TL02,2,30.5
1,A010,Agent_A010,TL02,2,34.5


In [0]:
print("Quality Compliance Report")

display(quality)

Quality Compliance Report


team_lead_id,agent_id,agent_name,quality_compliant_tickets
TL01,A001,Agent_A001,5
TL01,A004,Agent_A004,4
TL01,A002,Agent_A002,3
TL01,A003,Agent_A003,3
TL01,A005,Agent_A005,2
TL02,A009,Agent_A009,4
TL02,A010,Agent_A010,4
TL02,A007,Agent_A007,4
TL02,A006,Agent_A006,3
TL02,A008,Agent_A008,3


In [0]:
print("Day 2 Carry-over Report")

display(carryover)

Day 2 Carry-over Report


agent_id,ticket_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file,agent_name,role,team_lead_id,hours,minutes,seconds,resolution_time_minutes
A040,TKT00278,Resolved,0h 22m 0s,Technical,2,2026-07-18T19:38:05.777Z,day2_tickets.csv,Agent_A040,Junior Support Agent,TL08,0,22,0,22
A039,TKT00276,Resolved,0h 35m 45s,Returns,2,2026-07-18T19:38:05.777Z,day2_tickets.csv,Agent_A039,Support Agent,TL08,0,35,45,36
A040,TKT00279,Resolved,0h 29m 30s,Billing,2,2026-07-18T19:38:05.777Z,day2_tickets.csv,Agent_A040,Junior Support Agent,TL08,0,29,30,30
A038,TKT00274,Resolved,0h 26m 0s,Account,2,2026-07-18T19:38:05.777Z,day2_tickets.csv,Agent_A038,Senior Support Agent,TL08,0,26,0,26


## Pipeline Validation Summary

In [0]:
print("Pipeline Validation Completed Successfully!")

print("\nValidation Summary")
print("------------------------------")
print(f"Bronze Profiles : {bronze_profiles.count()}")
print(f"Bronze Day 1    : {bronze_day1.count()}")
print(f"Bronze Day 2    : {bronze_day2.count()}")

print(f"Silver Tickets  : {silver.count()}")

print(f"Team Report     : {team.count()}")
print(f"Agent Report    : {agent.count()}")
print(f"Quality Report  : {quality.count()}")
print(f"Carry-over      : {carryover.count()}")

print("\nAll validations passed successfully.")

Pipeline Validation Completed Successfully!

Validation Summary
------------------------------
Bronze Profiles : 42
Bronze Day 1    : 123
Bronze Day 2    : 86
Silver Tickets  : 77
Team Report     : 16
Agent Report    : 77
Quality Report  : 40
Carry-over      : 4

All validations passed successfully.
